# Container pack: full Cp / Cf / Cm pipeline

Runs the pressure pipeline on a multi-container body. Inputs are a body and probe XDMF+H5 timeseries (already in the new layout). The body is treated as a **single synthetic surface** -- no LNAS-level surface splitting and no manual sub-body authoring; region division is handled entirely by `ZoningModel` intervals derived automatically from the geometry (consecutive triangle-centroid gaps wider than `GAP_M` become interval boundaries).

**Guarantees:**
- The user's input body and probe H5 files are read-only -- nothing in this pipeline modifies them.
- Every output H5 file embeds the post-processing parameters under `/processing_metadata/` (timeseries) or `/{coef}/{cfg_lbl}[/{body}]/processing_metadata/` (results), so the file is self-describing for later debugging.
- All artifacts land flat in `./output/` -- no nested subfolders.

Outputs in `./output/` (flat layout):

* `pack.lnas` -- synthetic LNAS reconstructed from the body H5
* `cp.default.config.yaml`, `cp.default.time_series.{h5,xdmf}` -- Cp config + timeseries on the full mesh
* `Cf.containers.config.yaml`, `Cf.containers.pack.time_series.{h5,xdmf}` -- Cf config + timeseries with `cf_x/y/z` groups
* `Cm.containers.config.yaml`, `Cm.containers.pack.time_series.{h5,xdmf}` -- Cm config + timeseries with `cm_x/y/z` groups
* `results.{h5,xdmf}` -- combined per-coefficient stats with one Grid per leaf group

Cm uses `lever_strategy='region_base'`, so each container's overturning moment is computed about its own auto-derived base `(mean_x, mean_y, min_z)`. To override individual containers (e.g. for HFPI-style externally-known centers of mass), set `region_lever_origins={region_int: (x, y, z), ...}` in the Cm body config below.

In [1]:
import pathlib

import h5py
import numpy as np
from lnas import LnasFormat, LnasGeometry

from cfdmod.io.geometry.transformation_config import TransformationConfig
from cfdmod.pressure import (
    CfCaseConfig,
    CmCaseConfig,
    CpCaseConfig,
    run_cf,
    run_cm,
    run_cp,
)
from cfdmod.pressure.parameters import (
    BasicStatisticModel,
    BodyConfig,
    BodyDefinition,
    CfConfig,
    CmConfig,
    CpConfig,
    ExtremeAbsoluteParamsModel,
    MeanEquivalentParamsModel,
    MomentBodyConfig,
    ParameterizedStatisticModel,
    ZoningModel,
)
from cfdmod.utils import save_yaml

## Inputs and constants

Tweak these to match the case at hand. Defaults assume the body and probe sit at the repo root.

In [2]:
REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
BODY_H5 = REPO_ROOT / "bodies.body_cp body.h5"
PROBE_H5 = REPO_ROOT / "points.point_cp ref.h5"
OUTPUT = REPO_ROOT / "output"

GAP_M = 1.0  # min xy/z gap (metres) between containers; used to split the pack
SURFACE_NAME = "pack"  # synthetic single-surface label inside the LNAS

# Time window applied to Cp (and therefore propagated to Cf/Cm via cp.time_series.h5).
# Set either bound to None to fall back to the body H5's own range.
# Example below crops the warm-up: only timesteps with t >= 150 s are processed.
T_MIN: float | None = 150.0
T_MAX: float | None = None

# Flow / scaling -- placeholders, replace with the real values for this run.
SIMUL_U_H = 1.0
SIMUL_CHARACTERISTIC_LENGTH = 10.0
FLUID_DENSITY = 1.225
MACROSCOPIC_TYPE = "rho"  # 'rho' applies cs^2=1/3, 'pressure' leaves values as-is
REFERENCE_PRESSURE = "average"

# Stats applied to every coefficient (Cp, Cf, Cm)
STATISTICS = [
    BasicStatisticModel(stats="mean"),
    BasicStatisticModel(stats="rms"),
    BasicStatisticModel(stats="skewness"),
    BasicStatisticModel(stats="kurtosis"),
    ParameterizedStatisticModel(stats="mean_eq", params=MeanEquivalentParamsModel(scale_factor=0.61)),
    ParameterizedStatisticModel(stats="min", params=ExtremeAbsoluteParamsModel()),
    ParameterizedStatisticModel(stats="max", params=ExtremeAbsoluteParamsModel()),
]

OUTPUT.mkdir(parents=True, exist_ok=True)
assert BODY_H5.exists(), BODY_H5
assert PROBE_H5.exists(), PROBE_H5
BODY_H5, PROBE_H5, OUTPUT

(PosixPath('/home/waine/Documents/Codigos/AeroSim/cfdmod/bodies.body_cp body.h5'),
 PosixPath('/home/waine/Documents/Codigos/AeroSim/cfdmod/points.point_cp ref.h5'),
 PosixPath('/home/waine/Documents/Codigos/AeroSim/cfdmod/output'))

## Build the synthetic single-surface LNAS

The pipeline asks for an LNAS path. We don't want the user to pre-author surfaces, so we just wrap the body H5's `/Triangles + /Geometry` into an LNAS with a single surface that includes every triangle. Region division happens later, via the `ZoningModel`.

In [3]:
def build_lnas_from_body_h5(body_h5: pathlib.Path, out_lnas: pathlib.Path) -> LnasFormat:
    with h5py.File(body_h5, "r") as f:
        triangles = f["Triangles"][:].astype(np.int32)
        vertices = f["Geometry"][:].astype(np.float64)
    geom = LnasGeometry(vertices=vertices, triangles=triangles)
    surfaces = {SURFACE_NAME: np.arange(len(triangles), dtype=np.int32)}
    fmt = LnasFormat(version="v0.5.2", geometry=geom, surfaces=surfaces)
    out_lnas.parent.mkdir(parents=True, exist_ok=True)
    fmt.to_file(out_lnas)
    return fmt


PACK_LNAS = OUTPUT / "pack.lnas"
mesh = build_lnas_from_body_h5(BODY_H5, PACK_LNAS)
len(mesh.geometry.triangles), len(mesh.geometry.vertices)

(83715, 43015)

## Detect container partition from the geometry (>`GAP_M` gap rule)

Sort triangle-centroid coordinates on each axis, find consecutive jumps wider than `GAP_M`, and put a boundary at each gap midpoint. Result is a `ZoningModel` ready to drop into the body config -- one cell per container, no surface authoring required.

In [4]:
def detect_intervals(coords: np.ndarray, gap: float = GAP_M) -> list[float]:
    s = np.sort(coords)
    diffs = np.diff(s)
    splits = np.where(diffs > gap)[0]
    boundaries = [float((s[i] + s[i + 1]) / 2) for i in splits]
    return [float("-inf"), *boundaries, float("inf")]


centroids = np.mean(mesh.geometry.triangle_vertices, axis=1)
zoning = ZoningModel(
    x_intervals=detect_intervals(centroids[:, 0]),
    y_intervals=detect_intervals(centroids[:, 1]),
    z_intervals=detect_intervals(centroids[:, 2]),
)
n_x = max(1, len(zoning.x_intervals) - 1)
n_y = max(1, len(zoning.y_intervals) - 1)
n_z = max(1, len(zoning.z_intervals) - 1)
print(f"x: {len(zoning.x_intervals) - 2} interior boundary(ies) -> {n_x} bin(s)")
print(f"y: {len(zoning.y_intervals) - 2} interior boundary(ies) -> {n_y} bin(s)")
print(f"z: {len(zoning.z_intervals) - 2} interior boundary(ies) -> {n_z} bin(s)")
print(f"total cells (some may be empty): {n_x * n_y * n_z}")

x: 3 interior boundary(ies) -> 4 bin(s)
y: 1 interior boundary(ies) -> 2 bin(s)
z: 0 interior boundary(ies) -> 1 bin(s)
total cells (some may be empty): 8


## Build configs

Built in code (no YAML authoring). The Cm body uses `lever_strategy='region_base'` so each container's overturning moment is taken about its own base. To replace with externally-known centers of mass, pass `region_lever_origins={0: (x, y, z), 1: ...}` -- it overrides the strategy per region.

In [5]:
def detect_time_range(body_h5: pathlib.Path) -> tuple[float, float]:
    with h5py.File(body_h5, "r") as f:
        keys = list(f["pressure"].keys())
    times = sorted(float(k[1:]) for k in keys)
    return float(times[0]), float(times[-1])


detected_range = detect_time_range(BODY_H5)
timestep_range = (
    detected_range[0] if T_MIN is None else max(float(T_MIN), detected_range[0]),
    detected_range[1] if T_MAX is None else min(float(T_MAX), detected_range[1]),
)
print(f"detected:    [{detected_range[0]:.3f}, {detected_range[1]:.3f}]")
print(f"will use:    [{timestep_range[0]:.3f}, {timestep_range[1]:.3f}]")
assert timestep_range[0] < timestep_range[1], "empty time window"

cp_cfg = CpCaseConfig(
    pressure_coefficient={
        "default": CpConfig(
            statistics=STATISTICS,
            timestep_range=timestep_range,
            simul_U_H=SIMUL_U_H,
            simul_characteristic_length=SIMUL_CHARACTERISTIC_LENGTH,
            fluid_density=FLUID_DENSITY,
            macroscopic_type=MACROSCOPIC_TYPE,
            reference_pressure=REFERENCE_PRESSURE,
        )
    }
)

cf_cfg = CfCaseConfig(
    bodies={SURFACE_NAME: BodyDefinition(surfaces=[SURFACE_NAME])},
    force_coefficient={
        "containers": CfConfig(
            statistics=STATISTICS,
            bodies=[BodyConfig(name=SURFACE_NAME, sub_bodies=zoning)],
            directions=["x", "y", "z"],
            transformation=TransformationConfig(),
        )
    },
)

cm_cfg = CmCaseConfig(
    bodies={SURFACE_NAME: BodyDefinition(surfaces=[SURFACE_NAME])},
    moment_coefficient={
        "containers": CmConfig(
            statistics=STATISTICS,
            bodies=[
                MomentBodyConfig(
                    name=SURFACE_NAME,
                    sub_bodies=zoning,
                    lever_strategy="region_base",  # per-container base; replace via region_lever_origins for HFPI
                )
            ],
            directions=["x", "y", "z"],
            transformation=TransformationConfig(),
        )
    },
)

# The run_* functions take a YAML config path. We dump to a sidecar file used
# only as the input to the pipeline; the configs are also embedded inside the
# resulting H5 outputs under /processing_metadata, so this sidecar isn't the
# canonical record.
cp_cfg_path = OUTPUT / "_cp.config.in.yaml"
cf_cfg_path = OUTPUT / "_Cf.config.in.yaml"
cm_cfg_path = OUTPUT / "_Cm.config.in.yaml"
save_yaml(cp_cfg.model_dump(), cp_cfg_path)
save_yaml(cf_cfg.model_dump(), cf_cfg_path)
save_yaml(cm_cfg.model_dump(), cm_cfg_path)
cp_cfg_path, cf_cfg_path, cm_cfg_path

detected:    [119.978, 259.468]
will use:    [150.000, 259.468]


(PosixPath('/home/waine/Documents/Codigos/AeroSim/cfdmod/output/_cp.config.in.yaml'),
 PosixPath('/home/waine/Documents/Codigos/AeroSim/cfdmod/output/_Cf.config.in.yaml'),
 PosixPath('/home/waine/Documents/Codigos/AeroSim/cfdmod/output/_Cm.config.in.yaml'))

## Run Cp

Persists the Cp timeseries to `output/cp.default.time_series.{h5,xdmf}`, then computes statistics and writes them to `results.h5` under `/cp/default/`. The body and probe H5 files are not modified.

In [6]:
run_cp(
    body_h5=BODY_H5,
    probe_h5=PROBE_H5,
    mesh_path=PACK_LNAS,
    cfg_path=cp_cfg_path,
    output=OUTPUT,
)
CP_TIMESERIES = OUTPUT / "cp.default.time_series.h5"
CP_TIMESERIES

[2026-04-29 15:00:59,306] [INFO] - cfdmod - Processing Cp: default


[2026-04-29 15:00:59,307] [INFO] - cfdmod - Transforming to Cp timeseries...


[2026-04-29 15:01:00,077] [INFO] - cfdmod - Cp timeseries written to /home/waine/Documents/Codigos/AeroSim/cfdmod/output/cp.default.time_series.h5


[2026-04-29 15:01:00,080] [INFO] - cfdmod - Calculating Cp statistics from on-disk timeseries...


[2026-04-29 15:01:06,390] [INFO] - cfdmod - Cp stats written for config 'default'


PosixPath('/home/waine/Documents/Codigos/AeroSim/cfdmod/output/cp.default.time_series.h5')

## Run Cf

Streams Cp from the on-disk timeseries, applies the Cf transform per-region, broadcasts to the full pack mesh, and writes one `Cf.containers.pack.time_series.h5` with `cf_x`, `cf_y`, `cf_z` groups (each `t{T}` array has one value per triangle). Stats are computed from the on-disk file and appended to `results.h5` under `/cf_{dir}/containers/pack/`.

In [7]:
run_cf(
    cp_h5=CP_TIMESERIES,
    mesh_path=PACK_LNAS,
    cfg_path=cf_cfg_path,
    output=OUTPUT,
)

[2026-04-29 15:01:06,409] [INFO] - cfdmod - Processing Cf: containers


[2026-04-29 15:01:40,308] [INFO] - cfdmod - Cf stats written for config 'containers'


## Run Cm

Same disk-first contract; per-region overturning moment is taken about each region's auto-derived base. To plug externally-known centers of mass instead, edit the `cm_cfg` cell above to set `region_lever_origins={0: (...), 1: (...), ...}` keyed by integer region index.

In [8]:
run_cm(
    cp_h5=CP_TIMESERIES,
    mesh_path=PACK_LNAS,
    cfg_path=cm_cfg_path,
    output=OUTPUT,
)

[2026-04-29 15:01:40,326] [INFO] - cfdmod - Processing Cm: containers


[2026-04-29 15:02:09,539] [INFO] - cfdmod - Cm stats written for config 'containers'


## Inspect what landed

Walks `results.h5` and lists every leaf group with an embedded mesh + its stat datasets. Each group is its own `<Grid>` in `results.xdmf`, on the matching mesh -- no length mismatches.

In [9]:
with h5py.File(OUTPUT / "results.h5", "r") as f:
    rows = []
    def visit(name, obj):
        if isinstance(obj, h5py.Group) and "Triangles" in obj and "Geometry" in obj:
            n_tri = obj["Triangles"].shape[0]
            stats = sorted(k for k in obj if k not in ("Triangles", "Geometry"))
            rows.append((name, n_tri, stats))
    f.visititems(visit)
for name, n_tri, stats in rows:
    print(f"/{name}: n_tri={n_tri}, stats={stats}")

/cf_x/containers/pack: n_tri=83715, stats=['kurtosis', 'max', 'mean', 'mean_eq', 'min', 'processing_metadata', 'rms', 'skewness']
/cf_y/containers/pack: n_tri=83715, stats=['kurtosis', 'max', 'mean', 'mean_eq', 'min', 'processing_metadata', 'rms', 'skewness']
/cf_z/containers/pack: n_tri=83715, stats=['kurtosis', 'max', 'mean', 'mean_eq', 'min', 'processing_metadata', 'rms', 'skewness']
/cm_x/containers/pack: n_tri=83715, stats=['kurtosis', 'max', 'mean', 'mean_eq', 'min', 'processing_metadata', 'rms', 'skewness']
/cm_y/containers/pack: n_tri=83715, stats=['kurtosis', 'max', 'mean', 'mean_eq', 'min', 'processing_metadata', 'rms', 'skewness']
/cm_z/containers/pack: n_tri=83715, stats=['kurtosis', 'max', 'mean', 'mean_eq', 'min', 'processing_metadata', 'rms', 'skewness']
/cp/default: n_tri=83715, stats=['kurtosis', 'max', 'mean', 'mean_eq', 'min', 'processing_metadata', 'rms', 'skewness']


## Open in ParaView

Everything is flat under `./output/`:

* `output/results.xdmf` -- per-coefficient stats; one Grid per `(coefficient, direction, body)` triple, each on the matching mesh.
* `output/cp.default.time_series.xdmf` -- Cp animation.
* `output/Cf.containers.pack.time_series.xdmf` -- Cf animation; pick `cf_x`, `cf_y`, or `cf_z` from the Attribute selector.
* `output/Cm.containers.pack.time_series.xdmf` -- Cm animation; same selector for `cm_x/y/z`.

## Inspect embedded metadata

Each output file carries the parameters used to produce it. Read them back with `cfdmod.io.xdmf.read_processing_metadata`:

```python
from cfdmod.io.xdmf import read_processing_metadata
md = read_processing_metadata(OUTPUT / "cp.default.time_series.h5", "/")
print(md["config"])       # full Cp config (statistics, simul_U_H, etc.)
print(md["produced_at"])  # ISO timestamp
print(md["body_h5"])      # path to the input that produced this file
```

For per-leaf-group metadata in `results.h5`, pass the group path:

```python
md = read_processing_metadata(OUTPUT / "results.h5", "cf_x/containers/pack")
```